# Fact-Grounded Food Captioning

## 1. Project overview

**Fanza Khairan Pratama · ML research project · 2026**

This notebook implements five-output image nutrition regression, structured fact preparation,
adaptation of pretrained ViT and T5 models, and evaluation against five comparison methods.
The grounded captioner uses ingredient names from metadata alongside estimated nutrients;
it does not recognize those ingredients independently. See the [project README](../README.md)
for the result summary and static explorer.

The tables in `results/` come from saved experiments on 507 test dishes. Running this notebook
again can produce different values because training, library versions, stable template
assignment, and checkpoint selection affect results. Fresh CNN checkpoint inference must
remain separate from those historical tables: the later best-checkpoint review produced
different CNN predictions. New outputs are written to the configured local run directory.

### Before running

1. Use Python 3.12 and install `requirements.txt` from the repository root. Select a PyTorch
   build compatible with your CPU or CUDA environment, then open this notebook in Jupyter.
2. Obtain the external [Nutrition5k dataset](https://github.com/google-research-datasets/Nutrition5k):
   `metadata/dish_metadata_cafe1.csv`, `metadata/dish_metadata_cafe2.csv`,
   `dish_ids/splits/rgb_train_ids.txt`, `dish_ids/splits/rgb_test_ids.txt`, and
   `imagery/realsense_overhead/<dish_id>/rgb.png`. The demo's WebP photos are display assets.
3. Set `NUTRITION5K_DIR` to the dataset root and `NUTRITION_RUN_DIR` to a local run directory.
   Defaults are `data/nutrition5k_dataset` and `runs/portfolio` under the repository root.
   The kernel may start from either the repository root or `notebooks/`; helper imports
   resolve correctly from both locations.
4. For checkpoint inference, supply compatible checkpoints under the run's `models/`:
   `cnn_best.pth`, `captioner_freeze.pth`, `captioner_e2e.pth`, `baseline2_t5only.pth`,
   `conventional_freeze.pth`, and `conventional_e2e.pth`. Weights and raw data are not bundled.
   Initial pretrained ViT/T5 loading, tokenization, and language evaluation may download assets.
5. Training is disabled by default. Explicitly set `TRAIN_MODELS=True` to train missing models;
   set `FORCE_RETRAIN=True` only when intentionally replacing checkpoints in a fresh run folder.
   Optional `NUTRITION_REFERENCE_CAPTIONS` points to an existing `captions.csv` to reuse references.

Code outputs are cleared for readability. The final section also shows saved examples without
model inference; run the imports cell and that section to inspect them without the raw dataset.

## 2. Imports and configuration

In [ ]:
import os, sys, json, re, random, warnings, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import autocast, GradScaler

import timm
from transformers import (ViTModel, T5Tokenizer, T5ForConditionalGeneration,
                          get_linear_schedule_with_warmup)
from transformers.modeling_outputs import BaseModelOutput
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import nltk



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"Device      : {device}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU         : {torch.cuda.get_device_name(0)} ({vram:.1f} GB)")
# Resolve helpers from either the repository root or notebooks/ kernel directory.
REPO_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'research_utils.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository root or notebooks/ directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from research_utils import (parse_nutrition5k_csv, template_index, parse_fact_string,
                            legacy_extract_numbers, fact_numbers, numeric_metrics)


In [ ]:
# Inputs stay outside version control; every run writes to its own directory.
DATA_DIR = Path(os.environ.get('NUTRITION5K_DIR', str(REPO_ROOT / 'data/nutrition5k_dataset'))).expanduser().resolve()
BASE_DIR = Path(os.environ.get('NUTRITION_RUN_DIR', str(REPO_ROOT / 'runs/portfolio'))).expanduser().resolve()
OUTPUTS_DIR = BASE_DIR / 'outputs'
MODELS_DIR = BASE_DIR / 'models'
IMAGE_DIR = DATA_DIR / 'imagery' / 'realsense_overhead'
METADATA_DIR = DATA_DIR / 'metadata'
SPLITS_DIR = DATA_DIR / 'dish_ids' / 'splits'
TRAIN_MODELS = False
FORCE_RETRAIN = False
REFERENCE_CAPTIONS = os.environ.get('NUTRITION_REFERENCE_CAPTIONS', '')

def require_training():
    if not TRAIN_MODELS:
        raise RuntimeError('Training is disabled. Supply checkpoints or explicitly set TRAIN_MODELS=True in a fresh run directory.')

for folder in [IMAGE_DIR, METADATA_DIR, SPLITS_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(f'Missing dataset folder: {folder}; see README dataset setup.')
for folder in [OUTPUTS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ====== GLOBAL SWITCH ======
SEED          = 42
TARGET_COLS   = ['calories', 'protein', 'fat', 'carb', 'mass']
N_TARGETS     = len(TARGET_COLS)

# ====== IMAGE ======
IMAGE_SIZE    = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ====== CNN (Phase 3) ======
CNN_LR           = 1e-4
CNN_WEIGHT_DECAY = 1e-4
CNN_BATCH        = 32
CNN_MAX_EPOCHS   = 100
CNN_PATIENCE     = 7
CNN_SCHED_PAT    = 3
CNN_HIDDEN       = 256
CNN_DROPOUT      = 0.3
NUM_WORKERS      = 0  # Portable for notebook kernels, including Windows

# ====== CAPTIONER (Phase 5 / Baseline / Conventional) ======
VIT_ID            = 'google/vit-base-patch16-224-in21k'
T5_ID             = 't5-small'
CAP_LR            = 5e-5
CAP_WEIGHT_DECAY  = 0.01
CAP_BATCH         = 16
CAP_MAX_EPOCHS    = 50
CAP_PATIENCE      = 5
CAP_WARMUP_RATIO  = 0.1
CAP_GRAD_CLIP     = 1.0
USE_AMP           = device.type == "cuda"
BEAM_WIDTH        = 4
MAX_FACT_LENGTH   = 160   # dinaikkan dari 128 karena fact string memuat mass
MAX_CAPTION_LEN   = 96

print("[OK] Konfigurasi global siap")
print(f"     BASE_DIR    : {BASE_DIR}")
print(f"     OUTPUTS_DIR : {OUTPUTS_DIR}")
print(f"     MODELS_DIR  : {MODELS_DIR}")
print(f"     TARGET_COLS : {TARGET_COLS}")
print(f"     FORCE_RETRAIN: {FORCE_RETRAIN}")

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"[OK] Seed = {SEED}")

## 3. Data preparation

The historical CNN partition contains 2,478 training, 276 validation and 507 test dishes after
overhead-image filtering and one outlier removal. The official train/test assignment is preserved.
Target normalization uses training data only. The parser rejects malformed input.

In [ ]:
meta_cafe1 = parse_nutrition5k_csv(METADATA_DIR / 'dish_metadata_cafe1.csv')
meta_cafe2 = parse_nutrition5k_csv(METADATA_DIR / 'dish_metadata_cafe2.csv')
if set(meta_cafe1) & set(meta_cafe2):
    raise ValueError('Duplicate dishes between metadata files')
metadata_raw = {**meta_cafe1, **meta_cafe2}
print(f"Cafe 1       : {len(meta_cafe1)} dish")
print(f"Cafe 2       : {len(meta_cafe2)} dish")
print(f"Total merged : {len(metadata_raw)} dish")

In [ ]:
def load_ids(filepath):
    with open(filepath, 'r') as f:
        return [line.strip() for line in f if line.strip()]

train_ids_full = load_ids(SPLITS_DIR / 'rgb_train_ids.txt')
test_ids_all   = load_ids(SPLITS_DIR / 'rgb_test_ids.txt')
print(f"Official train: {len(train_ids_full)} | Official test: {len(test_ids_all)}")

# Filter ke dish yang punya rgb.png
valid_ids = set(d.name for d in IMAGE_DIR.iterdir()
                if d.is_dir() and (d / 'rgb.png').exists())
print(f"Dish dengan rgb.png valid: {len(valid_ids)}")

train_ids_filtered = [d for d in train_ids_full if d in valid_ids]
test_ids           = [d for d in test_ids_all   if d in valid_ids]

# Validation split 10% dari train
train_ids, val_ids = train_test_split(train_ids_filtered, test_size=0.1, random_state=SEED)
all_ids = train_ids + val_ids + test_ids

print(f"\nTrain : {len(train_ids)} | Val : {len(val_ids)} | Test : {len(test_ids)} | Total : {len(all_ids)}")
assert len(set(train_ids) & set(val_ids))  == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids)   & set(test_ids)) == 0
print("[OK] Tidak ada overlap antar split")

In [ ]:
# Cek coverage metadata
covered = sum(1 for d in all_ids if d in metadata_raw)
print(f"Dish dari split yang punya metadata: {covered}/{len(all_ids)}")

rows = []
for dish_id in all_ids:
    if dish_id not in metadata_raw:
        continue
    m = metadata_raw[dish_id]
    split = ('train' if dish_id in set(train_ids)
             else 'val' if dish_id in set(val_ids) else 'test')
    # Sort ingredients by mass (desc) untuk konsistensi
    ingr_sorted = sorted(m['ingredients'], key=lambda x: x['grams'], reverse=True)
    ingr_names  = [ing['name'] for ing in ingr_sorted]
    rows.append({
        'dish_id': dish_id, 'split': split,
        'calories': m['calories'], 'mass': m['mass'], 'fat': m['fat'],
        'carb': m['carb'], 'protein': m['protein'],
        'n_ingr': m['n_ingr'], 'ingredients': '|'.join(ingr_names),
    })

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
OUTLIER_THRESHOLDS = {
    'calories': (0, 3000), 'mass': (0, 2000), 'fat': (0, 300),
    'carb': (0, 500), 'protein': (0, 300),
}
print("=== Deteksi Outlier ===\n")
for col, (lo, hi) in OUTLIER_THRESHOLDS.items():
    mask = (df[col] < lo) | (df[col] > hi)
    print(f"{col:10s}: {mask.sum()} outlier (threshold {lo}-{hi})")
    if mask.sum() > 0:
        print(df[mask][['dish_id','split',col]].head().to_string(index=False))

# Buang outlier (semua atribut) + nilai negatif
n_before = len(df)
out_mask = pd.Series(False, index=df.index)
for col, (lo, hi) in OUTLIER_THRESHOLDS.items():
    out_mask |= (df[col] < lo) | (df[col] > hi)
out_mask |= (df[['calories','protein','fat','carb','mass']] < 0).any(axis=1)
removed = sorted(df[out_mask]['dish_id'].tolist())
df = df[~out_mask].reset_index(drop=True)
print(f"\n[OUTLIER] Dibuang {n_before - len(df)} dish")
if removed: print(f"          IDs: {removed}")
print(f"Dataset bersih: {df.shape}")

# Refresh split lists & metadata dict (5 atribut)
valid = set(df['dish_id'])
train_ids = [d for d in train_ids if d in valid]
val_ids   = [d for d in val_ids   if d in valid]
test_ids  = [d for d in test_ids  if d in valid]
metadata = {}
for _, r in df.iterrows():
    metadata[r['dish_id']] = {c: float(r[c]) for c in TARGET_COLS}
    metadata[r['dish_id']]['ingredients'] = r['ingredients']
print(f"Train {len(train_ids)} | Val {len(val_ids)} | Test {len(test_ids)}")

In [ ]:
# Training-only target statistics: log1p followed by standardization.
train_raw = np.array([[metadata[d][c] for c in TARGET_COLS] for d in train_ids], dtype=np.float64)
train_log = np.log1p(train_raw)
log_mean = torch.tensor(train_log.mean(axis=0), dtype=torch.float32)
log_std  = torch.tensor(train_log.std(axis=0),  dtype=torch.float32)
if not torch.all(log_std > 0):
    raise ValueError('Zero target standard deviation')
print("=== Statistik ruang LOG (training, 5 atribut) ===")
for i, c in enumerate(TARGET_COLS):
    print(f"  {c:9s}: log_mean={log_mean[i]:.4f}  log_std={log_std[i]:.4f}")

with open(OUTPUTS_DIR / 'target_stats_log.json', 'w') as f:
    json.dump({'mean': log_mean.tolist(), 'std': log_std.tolist(),
               'cols': TARGET_COLS, 'transform': 'log1p'}, f, indent=2)

# Simpan metadata bersih + split + statistik
df.to_csv(OUTPUTS_DIR / 'metadata_clean.csv', index=False)
df[list(TARGET_COLS)].describe().to_csv(OUTPUTS_DIR / 'metadata_stats.csv')
for name, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    with open(OUTPUTS_DIR / f'dish_ids_{name}.txt', 'w') as f:
        f.write('\n'.join(ids))



## 4. CNN nutrition prediction

A pretrained EfficientNetV2-B0 backbone produces a pooled feature vector, followed by a shared
256-unit layer and five scalar heads for calories, protein, fat, carbohydrates and mass.
Targets use `log1p` and training-only standardization. Training uses full precision and early
stopping on validation loss. Both fresh training and checkpoint reuse restore the best validation
checkpoint before inference. The historical tables remain unchanged; evaluating this checkpoint
again produces a new run, which can differ from the saved CNN predictions.

In [ ]:
class Nutrition5kCNNDataset(Dataset):
    """Dataset CNN 5 atribut. Transform: raw -> log1p -> z-score."""
    def __init__(self, dish_ids, metadata, image_dir, transform, log_mean, log_std):
        self.transform, self.image_dir = transform, Path(image_dir)
        self.log_mean, self.log_std = log_mean, log_std
        self.samples = []
        skipped = 0
        for d in dish_ids:
            if d not in metadata: skipped += 1; continue
            p = self.image_dir / d / 'rgb.png'
            if not p.exists(): skipped += 1; continue
            raw = torch.tensor([metadata[d][c] for c in TARGET_COLS], dtype=torch.float32)
            self.samples.append((d, p, raw))
        if skipped: print(f"  [Dataset] Skipped {skipped} dish")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        d, p, raw = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        tgt = (torch.log1p(raw) - self.log_mean) / self.log_std
        return img, tgt, d

cnn_train_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.RandomHorizontalFlip(0.5),
    T.RandomRotation(15), T.ColorJitter(0.3, 0.3, 0.2),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
cnn_eval_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

cnn_train_ds = Nutrition5kCNNDataset(train_ids, metadata, IMAGE_DIR, cnn_train_tf, log_mean, log_std)
cnn_val_ds   = Nutrition5kCNNDataset(val_ids,   metadata, IMAGE_DIR, cnn_eval_tf,  log_mean, log_std)
cnn_test_ds  = Nutrition5kCNNDataset(test_ids,  metadata, IMAGE_DIR, cnn_eval_tf,  log_mean, log_std)

def make_cnn_loader(ds, shuffle):
    return DataLoader(ds, batch_size=CNN_BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
cnn_train_loader = make_cnn_loader(cnn_train_ds, True)
cnn_val_loader   = make_cnn_loader(cnn_val_ds, False)
cnn_test_loader  = make_cnn_loader(cnn_test_ds, False)

print(f"Train {len(cnn_train_ds)} | Val {len(cnn_val_ds)} | Test {len(cnn_test_ds)}")
img, tgt, _ = next(iter(cnn_train_loader))
print(f"image {img.shape} | target {tgt.shape} | range [{tgt.min():.2f}, {tgt.max():.2f}]")

In [ ]:
class NutritionCNN(nn.Module):
    """
    Shared-then-Branch CNN untuk prediksi 5 atribut.
    Backbone : EfficientNetV2B0 (1280-dim GAP)
    Shared   : Linear(1280 -> 256) + ReLU + Dropout
    Branches : 5 x Linear(256 -> 1), satu per atribut
    Mengadaptasi pola multitask Thames (2021) & Yanuartanti (2025),
    kapasitas diperkecil untuk dataset terbatas.
    """
    def __init__(self, n_targets=N_TARGETS, hidden=CNN_HIDDEN, dropout=CNN_DROPOUT, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('tf_efficientnetv2_b0', pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        feat = self.backbone.num_features
        self.shared = nn.Sequential(nn.Linear(feat, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.heads  = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])
    def forward(self, x):
        h = self.shared(self.backbone(x))
        return torch.cat([hd(h) for hd in self.heads], dim=1)



In [ ]:
CNN_CKPT = MODELS_DIR / 'cnn_best.pth'
CNN_HIST = MODELS_DIR / 'cnn_history.json'

def cnn_train_epoch(model, loader, optimizer, criterion):
    model.train(); tot, n = 0., 0
    for imgs, tgts, _ in tqdm(loader, desc='Train', leave=False):
        imgs, tgts = imgs.to(device), tgts.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), tgts)
        loss.backward(); optimizer.step()
        tot += loss.item()*imgs.size(0); n += imgs.size(0)
    return tot/n

@torch.no_grad()
def cnn_validate(model, loader, criterion):
    model.eval(); tot, n = 0., 0
    for imgs, tgts, _ in tqdm(loader, desc='Val', leave=False):
        imgs, tgts = imgs.to(device), tgts.to(device)
        tot += criterion(model(imgs), tgts).item()*imgs.size(0); n += imgs.size(0)
    return tot/n

def train_cnn_full():
    require_training()
    model = NutritionCNN().to(device)
    optimizer = AdamW(model.parameters(), lr=CNN_LR, weight_decay=CNN_WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=CNN_SCHED_PAT)
    criterion = nn.MSELoss()
    best, wait = float('inf'), 0
    hist = {'train_loss': [], 'val_loss': [], 'lr': []}
    print(f"Training CNN (max {CNN_MAX_EPOCHS} epoch, patience {CNN_PATIENCE})...")
    for ep in range(1, CNN_MAX_EPOCHS+1):
        tr = cnn_train_epoch(model, cnn_train_loader, optimizer, criterion)
        vl = cnn_validate(model, cnn_val_loader, criterion)
        scheduler.step(vl)
        lr = optimizer.param_groups[0]['lr']
        hist['train_loss'].append(tr); hist['val_loss'].append(vl); hist['lr'].append(lr)
        mark = ''
        if vl < best:
            best, wait = vl, 0
            torch.save({'epoch':ep,'model_state':model.state_dict(),'val_loss':vl}, CNN_CKPT)
            mark = ' [BEST]'
        else: wait += 1
        print(f"Epoch {ep:3d} | train {tr:.4f} | val {vl:.4f} | lr {lr:.2e}{mark}")
        with open(CNN_HIST,'w') as f: json.dump(hist, f, indent=2)
        if wait >= CNN_PATIENCE: print(f"Early stop (patience {CNN_PATIENCE})"); break
    best_checkpoint = torch.load(CNN_CKPT, map_location=device, weights_only=True)
    model.load_state_dict(best_checkpoint['model_state'])
    return model

seed_everything()
if CNN_CKPT.exists() and not FORCE_RETRAIN:
    print("[SKIP] cnn_best.pth ada -> load")
    cnn_model = NutritionCNN(pretrained=False).to(device)
    ck = torch.load(CNN_CKPT, map_location=device, weights_only=True)
    cnn_model.load_state_dict(ck['model_state'])
    print(f"       epoch {ck['epoch']} | val {ck['val_loss']:.4f}")
else:
    require_training()
    cnn_model = train_cnn_full()
cnn_model.eval()
print("[OK] CNN siap")

In [ ]:
@torch.no_grad()
def predict_dataset(model, loader, log_mean, log_std):
    lm, ls = log_mean.to(device), log_std.to(device)
    P, Y, D = [], [], []
    model.eval()
    for imgs, tgts, dids in tqdm(loader, desc='Inferensi'):
        imgs = imgs.to(device)
        p = torch.clamp(torch.expm1(model(imgs)*ls + lm), min=0)
        y = torch.clamp(torch.expm1(tgts.to(device)*ls + lm), min=0)
        P.append(p.cpu().numpy()); Y.append(y.cpu().numpy()); D.extend(dids)
    return np.concatenate(P), np.concatenate(Y), D

test_preds, test_gts, _ = predict_dataset(cnn_model, cnn_test_loader, log_mean, log_std)
cnn_rows = []
for i, c in enumerate(TARGET_COLS):
    yt, yp = test_gts[:,i], test_preds[:,i]
    mae  = mean_absolute_error(yt, yp)
    rmse = math.sqrt(mean_squared_error(yt, yp))
    r2   = r2_score(yt, yp)
    cnn_rows.append({'Nutrient':c, 'MAE':round(mae,2), 'RMSE':round(rmse,2),
                     'R2':round(r2,3), 'Mean_GT':round(yt.mean(),2),
                     'MAE_pct':f'{mae/yt.mean()*100:.1f}%'})
cnn_metrics_df = pd.DataFrame(cnn_rows)
print("=== Evaluasi CNN (Test Set) ===")
print(cnn_metrics_df.to_string(index=False))
cnn_metrics_df.to_csv(OUTPUTS_DIR/'cnn_test_metrics.csv', index=False)
print("[OK] cnn_test_metrics.csv disimpan")

In [ ]:
all_ids = train_ids + val_ids + test_ids
cnn_full_ds     = Nutrition5kCNNDataset(all_ids, metadata, IMAGE_DIR, cnn_eval_tf, log_mean, log_std)
cnn_full_loader = make_cnn_loader(cnn_full_ds, False)
print(f"Inferensi seluruh {len(cnn_full_ds)} dish...")
P, Y, D = predict_dataset(cnn_model, cnn_full_loader, log_mean, log_std)
split_map = {**{d:'train' for d in train_ids}, **{d:'val' for d in val_ids}, **{d:'test' for d in test_ids}}
pred_df = pd.DataFrame({'dish_id': D, 'split': [split_map[d] for d in D]})
for i, c in enumerate(TARGET_COLS):
    pred_df[f'pred_{c}'] = P[:,i]
    pred_df[f'gt_{c}']   = Y[:,i]
pred_df.to_csv(OUTPUTS_DIR/'predictions.csv', index=False)
print(f"[OK] predictions.csv | shape {pred_df.shape}")
print(pred_df.head(3).to_string(index=False))
del cnn_model, cnn_full_ds, cnn_full_loader; torch.cuda.empty_cache()
print("[OK] Memori GPU dibersihkan")

## 5. Fact-string preparation

Training and validation facts use measured nutrients; test facts use CNN predictions rounded and
clamped to nonnegative integers. Ingredient names are supplied metadata sorted by ingredient mass.
Reference captions use eight templates filled with rounded measured nutrients.

New template assignments use stable SHA-256 selection. To retain existing reference text, set
`NUTRITION_REFERENCE_CAPTIONS` to the corresponding `captions.csv` before this section. Generating
new reference assignments changes the experiment.

In [ ]:
TEMPLATES = [
    "This dish ({mass}g) is composed of {ingredients}. It contains approximately {calories} calories, "
    "with {protein}g of protein, {fat}g of fat, and {carbs}g of carbohydrates.",

    "A {mass}g plate of {ingredients}, providing around {calories} calories. "
    "The macronutrient profile includes {protein}g protein, {fat}g fat, and {carbs}g carbohydrates.",

    "The meal features {ingredients} and weighs {mass}g, offering {calories} calories in total. "
    "It has {protein}g of protein, {fat}g of fat, and {carbs}g of carbohydrates.",

    "This {mass}g food item consists of {ingredients}. Nutritionally, it provides "
    "{calories} calories, {protein}g protein, {fat}g fat, and {carbs}g carbohydrates.",

    "Weighing {mass}g with {calories} calories, this dish of {ingredients} "
    "delivers {protein}g of protein, {fat}g of fat, and {carbs}g of carbohydrates.",

    "Served with {ingredients}, this {mass}g meal packs {calories} calories, "
    "{protein}g protein, {fat}g fat, and {carbs}g carbs.",

    "A {mass}g, {calories}-calorie dish made with {ingredients}, offering "
    "{protein}g protein, {fat}g fat, and {carbs}g carbohydrates.",

    "Prepared with {ingredients}, this {mass}g dish provides {calories} calories total, "
    "breaking down to {protein}g protein, {fat}g fat, and {carbs}g carbohydrates.",
]
print(f"[OK] {len(TEMPLATES)} template (semua memuat mass)")
for i, t in enumerate(TEMPLATES):
    print(f"  T{i+1}: {t[:85]}...")

In [ ]:
def fmt_ingredients(s):
    """Format pipe-separated ingredients ke English natural dengan Oxford comma."""
    if not isinstance(s, str) or not s.strip():
        return "various ingredients"
    items = [x.strip() for x in s.split("|") if x.strip()]
    if not items: return "various ingredients"
    if len(items) == 1: return items[0]
    if len(items) == 2: return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"

def build_fact_string(ingr, mass, cal, pro, fat, carb):
    return (f"ingredients: {ingr} | mass: {mass} g | calories: {cal} kcal | "
            f"protein: {pro} g | fat: {fat} g | carbohydrates: {carb} g")

def pick_template(dish_id):
    """Pilih template deterministik berdasarkan hash dish_id (konsisten saat rerun)."""
    return TEMPLATES[template_index(dish_id, len(TEMPLATES))]

print(fmt_ingredients("grilled chicken|steamed rice|broccoli|olive oil"))
print(build_fact_string("grilled chicken and rice", 300, 485, 38, 12, 52))

In [ ]:
pred_df_p4 = pd.read_csv(OUTPUTS_DIR/'predictions.csv')
pred_lookup = pred_df_p4.set_index('dish_id').to_dict('index')

old_caps = None
if REFERENCE_CAPTIONS:
    old_caps = pd.read_csv(REFERENCE_CAPTIONS).set_index('dish_id', verify_integrity=True)
    if set(old_caps.index) != set(train_ids + val_ids + test_ids):
        raise ValueError('Reference caption coverage differs from dataset')

fact_rows, cap_rows, n_placeholder, template_ids = [], [], 0, []
for dish_id in train_ids + val_ids + test_ids:
    if dish_id not in metadata: continue
    m, spl = metadata[dish_id], split_map[dish_id]
    ingr = fmt_ingredients(m['ingredients'])
    if ingr == "various ingredients": n_placeholder += 1
    # Fact string: test=prediksi CNN, train/val=GT
    if spl == 'test':
        if dish_id not in pred_lookup:
            raise ValueError(f'Missing CNN prediction for test dish {dish_id}')
        p = pred_lookup[dish_id]
        src = {c: max(0, round(p[f'pred_{c}'])) for c in TARGET_COLS}
    else:
        src = {c: max(0, round(m[c])) for c in TARGET_COLS}
    fact = build_fact_string(ingr, src['mass'], src['calories'], src['protein'], src['fat'], src['carb'])
    # Caption GT: selalu dari GT
    gt = {c: max(0, round(m[c])) for c in TARGET_COLS}
    tid = template_index(dish_id, len(TEMPLATES))
    template_ids.append(tid+1)
    caption = TEMPLATES[tid].format(ingredients=ingr, mass=gt['mass'], calories=gt['calories'],
                                     protein=gt['protein'], fat=gt['fat'], carbs=gt['carb'])
    if REFERENCE_CAPTIONS:
        old = old_caps.loc[dish_id]
        tid = int(old['template_id']) - 1
        if not 0 <= tid < len(TEMPLATES):
            raise ValueError(f'Invalid saved template for {dish_id}')
        expected_caption = TEMPLATES[tid].format(ingredients=ingr, mass=gt['mass'], calories=gt['calories'],
                                                protein=gt['protein'], fat=gt['fat'], carbs=gt['carb'])
        if old['caption'] != expected_caption:
            raise ValueError(f'Saved reference does not match this dataset: {dish_id}')
        caption = old['caption']
    fact_rows.append({'dish_id':dish_id, 'split':spl, 'fact_string':fact})
    cap_rows.append({'dish_id':dish_id, 'split':spl, 'template_id':tid+1, 'caption':caption})

facts_df = pd.DataFrame(fact_rows)
caps_df  = pd.DataFrame(cap_rows)
facts_df.to_csv(OUTPUTS_DIR/'fact_strings.csv', index=False)
caps_df.to_csv(OUTPUTS_DIR/'captions.csv', index=False)
print(f"[OK] fact_strings.csv {facts_df.shape} | captions.csv {caps_df.shape}")
print(f"Dish dengan 'various ingredients': {n_placeholder}")
print(f"\nSample fact   : {facts_df.iloc[0]['fact_string']}")
print(f"Sample caption: {caps_df.iloc[0]['caption']}")

In [ ]:
tok_check = T5Tokenizer.from_pretrained(T5_ID)
fact_tokens = [len(tok_check(fs)['input_ids']) for fs in facts_df['fact_string']]
cap_tokens  = [len(tok_check(c)['input_ids']) for c in caps_df['caption']]
print("=== Statistik Panjang (token) ===")
print(f"Fact   - mean {np.mean(fact_tokens):.0f} | P95 {np.percentile(fact_tokens,95):.0f} | max {max(fact_tokens)}")
print(f"Caption- mean {np.mean(cap_tokens):.0f}  | P95 {np.percentile(cap_tokens,95):.0f}  | max {max(cap_tokens)}")
print(f"MAX_FACT_LENGTH={MAX_FACT_LENGTH} -> {'AMAN' if np.percentile(fact_tokens,95) < MAX_FACT_LENGTH else 'NAIKKAN'}")
del tok_check
print('Fact strings beyond limit:', sum(n > MAX_FACT_LENGTH for n in fact_tokens))
print('Reference captions beyond limit:', sum(n > MAX_CAPTION_LEN for n in cap_tokens))
if max(fact_tokens) > MAX_FACT_LENGTH or max(cap_tokens) > MAX_CAPTION_LEN:
    warnings.warn('Some samples will be truncated. Changing the limits creates a new experimental setting.')


## 6. ViT + T5 captioning

The captioner adapts pretrained ViT-B/16 and T5-small. ViT supplies 197 visual tokens projected
from 768 to 512 dimensions; the T5 encoder represents the fact string. Concatenated fact and
visual representations condition the T5 decoder. The frozen variant keeps ViT in evaluation
mode; the trainable variant updates ViT as well. The CNN and discrete fact strings remain
outside the captioner's gradient path.

The retained captioning split is 2,202 training / 276 validation / 507 test dishes. Captioner
validation draws 276 dishes from the CNN training partition; the CNN's original validation
partition is unused here. Checkpoint reuse is the default, and training requires explicit activation.

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(T5_ID)
print(f"[OK] Tokenizer {T5_ID} | vocab {tokenizer.vocab_size}")

facts_df = pd.read_csv(OUTPUTS_DIR/'fact_strings.csv')
caps_df  = pd.read_csv(OUTPUTS_DIR/'captions.csv')
facts_map = facts_df.set_index('dish_id')['fact_string'].to_dict()
caps_map  = caps_df.set_index('dish_id')['caption'].to_dict()

# Val captioner dari sebagian train (test dipakai utuh utk evaluasi)
_shuf = train_ids[:]
random.Random(SEED).shuffle(_shuf)
cap_val_ids   = _shuf[:276]
cap_train_ids = _shuf[276:]
print(f"Captioner - train {len(cap_train_ids)} | val {len(cap_val_ids)} | test {len(test_ids)}")

In [ ]:
class CaptioningDataset(Dataset):
    """Dataset captioning: image + fact tokens (input) + caption tokens (label)."""
    def __init__(self, dish_ids, facts_map, caps_map, image_dir, transform, use_image=True):
        self.transform, self.image_dir, self.use_image = transform, Path(image_dir), use_image
        self.samples = []
        skipped = 0
        for d in dish_ids:
            if d not in facts_map or d not in caps_map: skipped += 1; continue
            if use_image and not (self.image_dir / d / 'rgb.png').exists():
                skipped += 1; continue
            self.samples.append((d, facts_map[d], caps_map[d]))
        if skipped: print(f"  [CapDS] skip {skipped}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        d, fact, cap = self.samples[idx]
        fenc = tokenizer(fact, max_length=MAX_FACT_LENGTH, padding='max_length',
                         truncation=True, return_tensors='pt')
        cenc = tokenizer(cap, max_length=MAX_CAPTION_LEN, padding='max_length',
                         truncation=True, return_tensors='pt')
        labels = cenc['input_ids'].squeeze(0).clone()
        labels[labels == tokenizer.pad_token_id] = -100
        item = {'input_ids': fenc['input_ids'].squeeze(0),
                'attention_mask': fenc['attention_mask'].squeeze(0),
                'labels': labels, 'dish_id': d}
        if self.use_image:
            item['image'] = self.transform(Image.open(self.image_dir / d / 'rgb.png').convert('RGB'))
        return item

cap_train_tf = T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)), T.RandomHorizontalFlip(0.5),
                          T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
cap_eval_tf  = T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)),
                          T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
_ds = CaptioningDataset([cap_train_ids[0]], facts_map, caps_map, IMAGE_DIR, cap_eval_tf)
_b = _ds[0]
print(f"image {_b['image'].shape} | input_ids {_b['input_ids'].shape} | labels {_b['labels'].shape}")
print("[OK] CaptioningDataset valid")

In [ ]:
class ViTT5Captioner(nn.Module):
    """
    Fact-grounded captioning: ViT-B/16 + T5-small.
    1. ViT ekstrak fitur visual (B,197,768) -> proj ke (B,197,512)
    2. T5 encoder ekstrak fitur fact string (B,L,512)
    3. Concat [fact, visual] -> T5 decoder generate caption
    use_fact=False -> conventional baseline (fact diabaikan).
    """
    def __init__(self, freeze_vit=True, use_fact=True):
        super().__init__()
        self.use_fact = use_fact
        self.freeze_vit = freeze_vit
        self.vit = ViTModel.from_pretrained(VIT_ID)
        self.t5  = T5ForConditionalGeneration.from_pretrained(T5_ID)
        self.visual_proj = nn.Linear(self.vit.config.hidden_size, self.t5.config.d_model)
        if freeze_vit:
            for p in self.vit.parameters(): p.requires_grad = False
            print(f"  ViT FROZEN ({sum(p.numel() for p in self.vit.parameters()):,} params)")
        else:
            print(f"  ViT TRAINABLE (end-to-end)")
    def train(self, mode=True):
        super().train(mode)
        if self.freeze_vit:
            self.vit.eval()
        return self

    def _encode(self, image, input_ids, attention_mask):
        vis = self.visual_proj(self.vit(pixel_values=image).last_hidden_state)
        dt = attention_mask.dtype if attention_mask is not None else torch.long
        vmask = torch.ones(vis.size(0), vis.size(1), device=vis.device, dtype=dt)
        if self.use_fact and input_ids is not None:
            fact = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            combined = torch.cat([fact, vis], dim=1)
            cmask    = torch.cat([attention_mask, vmask], dim=1)
        else:
            combined, cmask = vis, vmask
        return combined, cmask
    def forward(self, image, input_ids=None, attention_mask=None, labels=None):
        combined, cmask = self._encode(image, input_ids, attention_mask)
        return self.t5(encoder_outputs=BaseModelOutput(last_hidden_state=combined),
                       attention_mask=cmask, labels=labels)
    @torch.no_grad()
    def generate_caption(self, image, input_ids=None, attention_mask=None,
                         num_beams=BEAM_WIDTH, max_length=MAX_CAPTION_LEN):
        combined, cmask = self._encode(image, input_ids, attention_mask)
        return self.t5.generate(encoder_outputs=BaseModelOutput(last_hidden_state=combined),
                                attention_mask=cmask, num_beams=num_beams, max_length=max_length,
                                early_stopping=True, no_repeat_ngram_size=3)
print("[OK] ViTT5Captioner class defined")

In [ ]:
def cap_train_epoch(model, loader, optimizer, scheduler, scaler, params):
    model.train(); tot, n = 0., 0
    for b in tqdm(loader, desc='Train', leave=False):
        img = b['image'].to(device) if 'image' in b else None
        ii, am, lb = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
        optimizer.zero_grad()
        with autocast(device_type=device.type, enabled=USE_AMP):
            out = model(image=img, input_ids=ii, attention_mask=am, labels=lb)
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(params, CAP_GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tot += out.loss.item()*lb.size(0); n += lb.size(0)
    return tot/n

@torch.no_grad()
def cap_validate(model, loader):
    model.eval(); tot, n = 0., 0
    for b in tqdm(loader, desc='Val', leave=False):
        img = b['image'].to(device) if 'image' in b else None
        ii, am, lb = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
        with autocast(device_type=device.type, enabled=USE_AMP):
            out = model(image=img, input_ids=ii, attention_mask=am, labels=lb)
        tot += out.loss.item()*lb.size(0); n += lb.size(0)
    return tot/n

def train_captioner(model, tr_ids, va_ids, tag, use_image=True):
    require_training()
    ckpt = MODELS_DIR / f'{tag}.pth'
    hist_path = MODELS_DIR / f'{tag}_history.json'
    tr_ld = DataLoader(CaptioningDataset(tr_ids, facts_map, caps_map, IMAGE_DIR, cap_train_tf, use_image),
                       batch_size=CAP_BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    va_ld = DataLoader(CaptioningDataset(va_ids, facts_map, caps_map, IMAGE_DIR, cap_eval_tf, use_image),
                       batch_size=CAP_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = AdamW(params, lr=CAP_LR, weight_decay=CAP_WEIGHT_DECAY)
    total_steps = len(tr_ld) * CAP_MAX_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(CAP_WARMUP_RATIO*total_steps), total_steps)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    best, wait = float('inf'), 0
    hist = {'train_loss':[], 'val_loss':[], 'lr':[]}
    print(f"Training {tag} | trainable {sum(p.numel() for p in params):,} | steps {total_steps}")
    for ep in range(1, CAP_MAX_EPOCHS+1):
        tr = cap_train_epoch(model, tr_ld, opt, sched, scaler, params)
        vl = cap_validate(model, va_ld)
        lr = opt.param_groups[0]['lr']
        hist['train_loss'].append(tr); hist['val_loss'].append(vl); hist['lr'].append(lr)
        mark = ''
        if vl < best:
            best, wait = vl, 0; torch.save(model.state_dict(), ckpt); mark = ' [BEST]'
        else: wait += 1
        print(f"  ep{ep:2d}/{CAP_MAX_EPOCHS} | train {tr:.4f} | val {vl:.4f} | lr {lr:.2e}{mark}")
        with open(hist_path,'w') as f: json.dump(hist, f, indent=2)
        if wait >= CAP_PATIENCE: print(f"  Early stop (patience {CAP_PATIENCE})"); break
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return model

@torch.no_grad()
def generate_captions(model, dish_ids, use_image, tag):
    ld = DataLoader(CaptioningDataset(dish_ids, facts_map, caps_map, IMAGE_DIR, cap_eval_tf, use_image),
                    batch_size=CAP_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); rows = []
    for b in tqdm(ld, desc=f'Generate {tag}', leave=False):
        img = b['image'].to(device) if 'image' in b else None
        ii, am = b['input_ids'].to(device), b['attention_mask'].to(device)
        gen = model.generate_caption(image=img, input_ids=ii, attention_mask=am)
        for d, t in zip(b['dish_id'], tokenizer.batch_decode(gen, skip_special_tokens=True)):
            rows.append({'dish_id':d, 'fact_string':facts_map.get(d,''),
                         'ground_truth_caption':caps_map.get(d,''), 'generated_caption':t})
    return pd.DataFrame(rows)

def plot_cap_curve(tag, label):
    hp = MODELS_DIR / f'{tag}_history.json'
    if not hp.exists(): return
    with open(hp) as f: h = json.load(f)
    ep = range(1, len(h['train_loss'])+1)
    fig, ax = plt.subplots(1, 2, figsize=(14,4))
    ax[0].plot(ep, h['train_loss'], 'o-', color='#4A90D9', label='Train', ms=3)
    ax[0].plot(ep, h['val_loss'],   'o-', color='#E07B54', label='Val',   ms=3)
    ax[0].set_title(f'{label} — Loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(ep, h['lr'], 'o-', color='#6CBF8A', ms=3); ax[1].set_yscale('log')
    ax[1].set_title(f'{label} — LR'); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR/f'{tag}_curves.png', dpi=130, bbox_inches='tight'); plt.show()

def viz_sample_captions(gen_df, tag, n=6):
    seed_everything()
    idxs = random.sample(range(len(gen_df)), min(n, len(gen_df)))
    def wrap(t, w=55):
        words, lines, cur = t.split(), [], ''
        for x in words:
            if len(cur)+len(x) < w: cur += (' ' if cur else '')+x
            else: lines.append(cur); cur = x
        if cur: lines.append(cur)
        return '\n'.join(lines)
    fig, axes = plt.subplots(3, 2, figsize=(18, 16))
    for ax, idx in zip(axes.flat, idxs):
        r = gen_df.iloc[idx]
        p = IMAGE_DIR / r['dish_id'] / 'rgb.png'
        if p.exists(): ax.imshow(plt.imread(str(p)))
        ax.set_title(f"[{r['dish_id']}]\n\nGENERATED:\n{wrap(r['generated_caption'])}\n\nGT:\n{wrap(r['ground_truth_caption'])}",
                     fontsize=8, loc='left', pad=8)
        ax.axis('off')
    plt.suptitle(f'Caption Results — {tag}', fontsize=13, fontweight='bold', y=1.0)
    plt.tight_layout(); plt.savefig(OUTPUTS_DIR/f'{tag}_samples.png', dpi=120, bbox_inches='tight'); plt.show()

print("[OK] Fungsi training/generasi/visualisasi captioner siap")

In [ ]:
# Run frozen and trainable variants; missing weights require explicit training activation.
for FREEZE in [True, False]:
    tag = 'captioner_freeze' if FREEZE else 'captioner_e2e'
    out_csv = OUTPUTS_DIR / f'test_captions_{"freeze" if FREEZE else "e2e"}.csv'
    ckpt = MODELS_DIR / f'{tag}.pth'
    print(f"\n{'='*60}\nPhase 5: {tag.upper()} (freeze_vit={FREEZE})\n{'='*60}")
    seed_everything()
    model = ViTT5Captioner(freeze_vit=FREEZE, use_fact=True).to(device)
    if ckpt.exists() and not FORCE_RETRAIN:
        print(f"[SKIP] {ckpt.name} ada -> load")
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    else:
        model = train_captioner(model, cap_train_ids, cap_val_ids, tag, use_image=True)
    plot_cap_curve(tag, tag)
    gen_df = generate_captions(model, test_ids, use_image=True, tag=tag)
    gen_df.to_csv(out_csv, index=False)
    print(f"[OK] {out_csv.name} ({len(gen_df)})")
    viz_sample_captions(gen_df, tag)
    del model; torch.cuda.empty_cache()

## 7. Baselines and evaluation

### 7.1 Template Fill and T5-only

Template filling copies supplied numbers into the selected reference template. Its text-overlap
advantage reflects shared formatting. T5-only adapts the pretrained model to read the fact string
with a `generate caption:` prefix and receives no image.

In [ ]:
# Strict parser is imported from research_utils. Missing facts are errors, never zeros.
print(parse_fact_string(facts_map[test_ids[0]]))

In [ ]:
# Baseline 1: Template Fill (near-oracle untuk NutAcc)
b1_rows = []
for d in test_ids:
    if d not in facts_map: continue
    f = parse_fact_string(facts_map[d])
    tid = int(caps_df.set_index('dish_id').loc[d, 'template_id']) - 1
    cap = TEMPLATES[tid].format(ingredients=f['ingredients'], mass=f['mass'],
            calories=f['calories'], protein=f['protein'], fat=f['fat'], carbs=f['carbohydrates'])
    b1_rows.append({'dish_id':d, 'fact_string':facts_map[d],
                    'ground_truth_caption':caps_map.get(d,''), 'generated_caption':cap})
df_b1 = pd.DataFrame(b1_rows)
df_b1.to_csv(OUTPUTS_DIR/'test_captions_baseline1.csv', index=False)
print(f"[OK] baseline1: {len(df_b1)} captions")
for i in range(2):
    print(f"\n  GT: {df_b1.iloc[i]['ground_truth_caption'][:100]}")
    print(f"  B1: {df_b1.iloc[i]['generated_caption'][:100]}")

In [ ]:
class FactOnlyDataset(Dataset):
    """Baseline 2: hanya fact string, tanpa gambar."""
    def __init__(self, dish_ids, facts_map, caps_map):
        self.samples = [(d, facts_map[d], caps_map[d]) for d in dish_ids
                        if d in facts_map and d in caps_map]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        d, fact, cap = self.samples[idx]
        fenc = tokenizer(f"generate caption: {fact}", max_length=MAX_FACT_LENGTH,
                         padding='max_length', truncation=True, return_tensors='pt')
        cenc = tokenizer(cap, max_length=MAX_CAPTION_LEN, padding='max_length',
                         truncation=True, return_tensors='pt')
        labels = cenc['input_ids'].squeeze(0).clone()
        labels[labels == tokenizer.pad_token_id] = -100
        return {'input_ids': fenc['input_ids'].squeeze(0),
                'attention_mask': fenc['attention_mask'].squeeze(0),
                'labels': labels, 'dish_id': d}

class T5OnlyCaptioner(nn.Module):
    """T5 tanpa ViT: fact string -> caption."""
    def __init__(self):
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(T5_ID)
    def forward(self, image=None, input_ids=None, attention_mask=None, labels=None):
        return self.t5(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    @torch.no_grad()
    def generate_caption(self, image=None, input_ids=None, attention_mask=None,
                         num_beams=BEAM_WIDTH, max_length=MAX_CAPTION_LEN):
        return self.t5.generate(input_ids=input_ids, attention_mask=attention_mask,
                                num_beams=num_beams, max_length=max_length,
                                early_stopping=True, no_repeat_ngram_size=3)

# T5-only butuh dataset & loader sendiri (FactOnlyDataset). Override sementara generate.
def train_baseline2(tag='baseline2_t5only'):
    ckpt = MODELS_DIR / f'{tag}.pth'
    hist_path = MODELS_DIR / f'{tag}_history.json'
    model = T5OnlyCaptioner().to(device)
    if ckpt.exists() and not FORCE_RETRAIN:
        print(f"[SKIP] {tag} ada -> load")
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        return model
    require_training()
    tr_ld = DataLoader(FactOnlyDataset(cap_train_ids, facts_map, caps_map),
                       batch_size=CAP_BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    va_ld = DataLoader(FactOnlyDataset(cap_val_ids, facts_map, caps_map),
                       batch_size=CAP_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    params = list(model.parameters())
    opt = AdamW(params, lr=CAP_LR, weight_decay=CAP_WEIGHT_DECAY)
    total_steps = len(tr_ld)*CAP_MAX_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(CAP_WARMUP_RATIO*total_steps), total_steps)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    best, wait = float('inf'), 0
    hist = {'train_loss':[], 'val_loss':[], 'lr':[]}
    print(f"Training {tag} | params {sum(p.numel() for p in params):,}")
    for ep in range(1, CAP_MAX_EPOCHS+1):
        model.train(); tot, n = 0., 0
        for b in tqdm(tr_ld, desc=f'{tag} ep{ep}', leave=False):
            ii, am, lb = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
            opt.zero_grad()
            with autocast(device_type=device.type, enabled=USE_AMP):
                out = model(input_ids=ii, attention_mask=am, labels=lb)
            scaler.scale(out.loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, CAP_GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            tot += out.loss.item()*lb.size(0); n += lb.size(0)
        tr = tot/n
        model.eval(); tot, n = 0., 0
        with torch.no_grad():
            for b in va_ld:
                ii, am, lb = b['input_ids'].to(device), b['attention_mask'].to(device), b['labels'].to(device)
                with autocast(device_type=device.type, enabled=USE_AMP):
                    out = model(input_ids=ii, attention_mask=am, labels=lb)
                tot += out.loss.item()*lb.size(0); n += lb.size(0)
        vl = tot/n; lr = opt.param_groups[0]['lr']
        hist['train_loss'].append(tr); hist['val_loss'].append(vl); hist['lr'].append(lr)
        mark = ''
        if vl < best: best, wait = vl, 0; torch.save(model.state_dict(), ckpt); mark=' [BEST]'
        else: wait += 1
        print(f"  ep{ep:2d} | train {tr:.4f} | val {vl:.4f}{mark}")
        with open(hist_path,'w') as f: json.dump(hist, f, indent=2)
        if wait >= CAP_PATIENCE: print("  Early stop"); break
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return model

@torch.no_grad()
def generate_baseline2(model, dish_ids):
    ld = DataLoader(FactOnlyDataset(dish_ids, facts_map, caps_map),
                    batch_size=CAP_BATCH, shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); rows = []
    for b in tqdm(ld, desc='gen baseline2', leave=False):
        ii, am = b['input_ids'].to(device), b['attention_mask'].to(device)
        gen = model.generate_caption(input_ids=ii, attention_mask=am)
        for d, t in zip(b['dish_id'], tokenizer.batch_decode(gen, skip_special_tokens=True)):
            rows.append({'dish_id':d, 'fact_string':facts_map.get(d,''),
                         'ground_truth_caption':caps_map.get(d,''), 'generated_caption':t})
    return pd.DataFrame(rows)

seed_everything()
b2_model = train_baseline2()
plot_cap_curve('baseline2_t5only', 'Baseline 2 T5-only')
df_b2 = generate_baseline2(b2_model, test_ids)
df_b2.to_csv(OUTPUTS_DIR/'test_captions_baseline2.csv', index=False)
print(f"[OK] baseline2: {len(df_b2)} captions")
del b2_model; torch.cuda.empty_cache()

### 7.2 Image-only ViT + T5

The frozen and trainable image-only models use the same captioner with `use_fact=False`.
They receive neither nutrition estimates nor ingredient metadata, so the comparison changes
available information as well as the conditioning mechanism.

In [ ]:
# Conventional: ViT+T5 TANPA fact grounding (use_fact=False), loop freeze+e2e
for FREEZE in [True, False]:
    tag = 'conventional_freeze' if FREEZE else 'conventional_e2e'
    out_csv = OUTPUTS_DIR / f'test_captions_{tag}.csv'
    ckpt = MODELS_DIR / f'{tag}.pth'
    print(f"\n{'='*60}\nConventional: {tag.upper()} (use_fact=False)\n{'='*60}")
    seed_everything()
    model = ViTT5Captioner(freeze_vit=FREEZE, use_fact=False).to(device)
    if ckpt.exists() and not FORCE_RETRAIN:
        print(f"[SKIP] {ckpt.name} ada -> load")
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    else:
        model = train_captioner(model, cap_train_ids, cap_val_ids, tag, use_image=True)
    plot_cap_curve(tag, tag)
    gen_df = generate_captions(model, test_ids, use_image=True, tag=tag)
    gen_df.to_csv(out_csv, index=False)
    print(f"[OK] {out_csv.name} ({len(gen_df)})")
    print("  Preview (angka = TEBAKAN dari gambar, bukan fakta):")
    for i in range(min(2, len(gen_df))):
        print(f"  GT : {gen_df.iloc[i]['ground_truth_caption'][:100]}")
        print(f"  GEN: {gen_df.iloc[i]['generated_caption'][:100]}")
    viz_sample_captions(gen_df, tag)
    del model; torch.cuda.empty_cache()

### 7.3 Evaluation

Numerical faithfulness (NutAcc) uses the historical integer regex and absolute ±1 tolerance
against supplied fact values. “Omit” means extraction failure, which can reflect formatting
or an absent attribute. The historical mass fallback is permissive.

Caption nutrient MAE uses measured nutrition only where extraction succeeds; interpret it
alongside omission rates. Language evaluation uses template references. CIDEr requires
`pycocoevalcap`; BERTScore may download a large model. All generated tables are written to
the local run directory, separate from the saved results in the repository.

In [ ]:
for resource in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f'Could not download NLTK resource: {resource}')

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer as _rs
from bert_score import score as bert_score_fn
from pycocoevalcap.cider.cider import Cider as CiderScorer

rscorer = _rs.RougeScorer(['rougeL'], use_stemmer=True)
smooth  = SmoothingFunction().method1

def calc_bleu(refs, hyps):
    rt = [[r.lower().split()] for r in refs]; ht = [h.lower().split() for h in hyps]
    return {'BLEU-1': corpus_bleu(rt, ht, weights=(1,0,0,0), smoothing_function=smooth),
            'BLEU-2': corpus_bleu(rt, ht, weights=(.5,.5,0,0), smoothing_function=smooth),
            'BLEU-3': corpus_bleu(rt, ht, weights=(1/3,1/3,1/3,0), smoothing_function=smooth),
            'BLEU-4': corpus_bleu(rt, ht, weights=(.25,.25,.25,.25), smoothing_function=smooth)}

def calc_cider(refs, hyps, ids):
    rd = {d:[r] for d,r in zip(ids,refs)}
    hd = {d:[h] for d,h in zip(ids,hyps)}
    return float(CiderScorer().compute_score(rd, hd)[0])

def calc_meteor(refs, hyps):
    return float(np.mean([meteor_score([word_tokenize(r)], word_tokenize(h)) for r,h in zip(refs,hyps)]))
def calc_rougeL(refs, hyps):
    return float(np.mean([rscorer.score(r,h)['rougeL'].fmeasure for r,h in zip(refs,hyps)]))
def calc_bertscore(refs, hyps):
    P,R,F1 = bert_score_fn(hyps, refs, lang='en', verbose=False, rescale_with_baseline=True)
    return {'BERTScore-P':float(P.mean()), 'BERTScore-R':float(R.mean()), 'BERTScore-F1':float(F1.mean())}
print("[OK] Fungsi metrik linguistik siap")

In [ ]:
extract_nums = legacy_extract_numbers
fact_nums = fact_numbers

In [ ]:
MODEL_FILES = {
    'Baseline 1 (Template Fill)':  'test_captions_baseline1.csv',
    'Baseline 2 (T5-only)':        'test_captions_baseline2.csv',
    'ViT+T5 Freeze':               'test_captions_freeze.csv',
    'ViT+T5 E2E':                  'test_captions_e2e.csv',
    'Conventional Freeze':         'test_captions_conventional_freeze.csv',
    'Conventional E2E':            'test_captions_conventional_e2e.csv',
}
gt_phys = {d: {c: metadata[d][c] for c in TARGET_COLS} for d in test_ids if d in metadata}
ref_map = caps_df.set_index('dish_id')['caption'].to_dict()

ling_rows, nut_rows, gt_rows, mismatch_data = [], [], [], {}
for name, fn in MODEL_FILES.items():
    fp = OUTPUTS_DIR / fn
    if not fp.exists(): raise FileNotFoundError(fp)
    gdf = pd.read_csv(fp)
    gmap = gdf.set_index('dish_id', verify_integrity=True)['generated_caption'].to_dict()
    ids = [d for d in test_ids if d in gmap and d in ref_map and d in facts_map]
    if set(ids) != set(test_ids) or set(gmap) != set(test_ids):
        raise ValueError(f'Incomplete test coverage: {name}')
    refs = [ref_map[d] for d in ids]; hyps = [str(gmap[d]) for d in ids]
    print(f"\n=== {name} ({len(ids)} dish) ===")
    bleu = calc_bleu(refs, hyps); cider = calc_cider(refs, hyps, ids)
    meteor = calc_meteor(refs, hyps); rougeL = calc_rougeL(refs, hyps)
    bscore = calc_bertscore(refs, hyps)
    ling_rows.append({'Model':name, **{k:round(v,4) for k,v in bleu.items()},
                      'CIDEr':round(cider,4), 'METEOR':round(meteor,4),
                      'ROUGE-L':round(rougeL,4), **{k:round(v,4) for k,v in bscore.items()}})
    print(f"  BLEU-4={bleu['BLEU-4']:.4f} CIDEr={cider:.4f} METEOR={meteor:.4f} ROUGE-L={rougeL:.4f} BERTScore-F1={bscore['BERTScore-F1']:.4f}")
    nut_ok = {c:0 for c in TARGET_COLS}; nut_tot = {c:0 for c in TARGET_COLS}; mism=[]
    for d in ids:
        gn = extract_nums(gmap[d]); fnv = fact_nums(facts_map[d]); ok={}
        for c in TARGET_COLS:
            nut_tot[c]+=1
            good = gn[c] is not None and abs(gn[c]-fnv[c])<=1
            ok[c]=good
            if good: nut_ok[c]+=1
        if not all(ok.values()): mism.append({'dish_id':d,'gen':gn,'fact':fnv})
    mismatch_data[name]=mism
    nut_rows.append({'Model':name, **{f'NutAcc_{c}':round(100*nut_ok[c]/nut_tot[c],2) for c in TARGET_COLS},
                     'NutAcc_avg':round(100*sum(nut_ok.values())/sum(nut_tot.values()),2)})
    print(f"  NutAcc avg={nut_rows[-1]['NutAcc_avg']:.1f}% | mismatch {len(mism)}/{len(ids)}")
    errs={c:[] for c in TARGET_COLS}; omits={c:0 for c in TARGET_COLS}
    for d in ids:
        gn = extract_nums(gmap[d]); gt = gt_phys.get(d,{})
        for c in TARGET_COLS:
            if gn[c] is None: omits[c]+=1
            elif c in gt: errs[c].append(abs(gn[c]-gt[c]))
    gt_rows.append({'Model':name,
                    **{f'MAE_{c}':round(np.mean(errs[c]),2) if errs[c] else None for c in TARGET_COLS},
                    **{f'Omit_{c}':round(100*omits[c]/len(ids),1) for c in TARGET_COLS}})
    print(f"  MAE-GT cal={gt_rows[-1]['MAE_calories']} mass={gt_rows[-1]['MAE_mass']}")
ling_df = pd.DataFrame(ling_rows); nut_df = pd.DataFrame(nut_rows); gt_df = pd.DataFrame(gt_rows)
print("\n[OK] Evaluasi 6 model selesai")

In [ ]:
final_df = ling_df.merge(nut_df, on='Model').merge(gt_df, on='Model')
final_df.to_csv(OUTPUTS_DIR/'eval_final.csv', index=False)
print("=== LINGUISTIK ===")
print(ling_df.to_string(index=False))
print("\n=== NUTACC (caption vs fact string) ===")
print(nut_df.to_string(index=False))
print("\n=== METRIK 2 (MAE caption vs GT fisik) ===")
print(gt_df[['Model']+[f'MAE_{c}' for c in TARGET_COLS]+['Omit_calories']].to_string(index=False))
print("\n[OK] eval_final.csv disimpan")

## 8. Examples and interpretation

The following examples read the same stored captions and nutrients as the demo and do not run
inference. After running the imports cell, they can be inspected independently of the dataset
and training sections. These examples and tables describe the historical experiment only.

Numerical faithfulness means agreement with supplied estimates, not physical accuracy. Ingredient
names are metadata, and template references favor Template Fill. T5-only is competitive, so the
scores do not demonstrate a statistically established visual advantage. Multi-seed replication
and confidence intervals are outside the available results.

In [ ]:
# Inspect historical outputs without loading model checkpoints or the raw dataset.
from IPython.display import display

archive_text = (REPO_ROOT / 'docs/assets/js/archive-data.js').read_text(encoding='utf-8')
saved_archive = json.loads(archive_text.split('=', 1)[1].strip().removesuffix(';'))
print(f"{len(saved_archive['rows'])} dishes; {sum(len(row['c']) for row in saved_archive['rows'])} saved captions")
display(pd.read_csv(REPO_ROOT / 'results/eval_final.csv')[['Model', 'BLEU-4', 'CIDEr', 'NutAcc_avg']])
print('Historical CNN results (separate from any new checkpoint inference):')
display(pd.read_csv(REPO_ROOT / 'results/cnn_test_metrics.csv'))

for dish_id in ['dish_1566501575', 'dish_1566838286']:
    row = next(item for item in saved_archive['rows'] if item['id'] == dish_id)
    print(f"\n{row['id']} — {row['i']}")
    with Image.open(REPO_ROOT / 'docs' / row['photo']) as photo:
        display(photo.copy())
    display(pd.DataFrame({'Nutrient': saved_archive['fields'],
                          'Supplied': row['f'], 'Measured': row['gt']}))
    for code in ['grounded', 'conventional']:
        j = next(i for i, model in enumerate(saved_archive['models']) if model['code'] == code)
        print(f"{saved_archive['models'][j]['label']}: {row['c'][j]}")
